# High-Level API - One-Liner ML

TuiML's high-level API lets you **train, evaluate, compare, save, and serve** models with single function calls. No boilerplate, no manual splitting, no metric plumbing.

`train()` takes **one spec dict** and returns a **fitted `Workflow`**: the pipeline *is* the model, so prediction, evaluation, persistence, and serving are methods on the object you get back.

TuiML has two doors into the same engine. `train()` is the **declarative door**: everything is dicts, serializable to JSON, made for configs and agents. `Workflow` is the **object door**: imported classes and configured instances, made for autocomplete and type checking. One notation per door, no overlap.

| Function | Purpose |
|----------|---------|
| `tuiml.train()` | Train and evaluate a model from one spec dict (or a `.json` path) |
| `tuiml.Benchmark` | Compare models across datasets |
| `tuiml.list_algorithms()` | Discover available algorithms |
| `tuiml.describe_algorithm()` | Get parameter schema for an algorithm |
| `tuiml.search_algorithms()` | Search algorithms by keyword |
| `tuiml.serve()` / `stop_server()` | Serve a model via REST API |

| Method on the returned model | Purpose |
|------------------------------|---------|
| `model.predict(X)` / `model.predict_proba(X)` | Make predictions |
| `model.evaluate(X, y)` / `model.score(X, y)` | Evaluate on new data |
| `model.save(path)` / `Workflow.load(path)` | Persist and restore |
| `model.serve(port=...)` | Serve this pipeline over REST |

> 🤖 **Agent equivalent:** every function on this page is also a typed tool. `tuiml.train(...)` ↔ `tuiml_train(...)`, `tuiml.Benchmark(...).run()` ↔ `tuiml_benchmark(...)`, `model.predict(...)` ↔ `tuiml_predict(...)`. An agent connected via MCP calls these exact tools with the same arguments. See [Track A](/docs/tutorials.html#track-agents) to wire one up.

In [1]:
import tuiml
import numpy as np

[tuiml] loaded 2 user algorithm(s)


## 1. `tuiml.train()` - The Core One-Liner

`train()` takes exactly one argument: a **spec dict** (or a path to a `.json` file holding it). The two required keys are `"model"` and `"data"`. The model spec is always `{"name": ..., "params": {...}}` (`"params"` optional); the data spec names a `source` (a builtin dataset name or a file path) and the `target` column.

What comes back is a fitted `Workflow`. Its results live on trailing-underscore attributes: `metrics_`, `model_`, `cv_results_`, `metadata_`.

In [2]:
# Simplest possible call - train on built-in iris dataset
model = tuiml.train({
    "model": {"name": "RandomForestClassifier"},
    "data": {"source": "iris", "target": "class"},
})

print(type(model))                                # Workflow (fitted)
print("Metrics:", model.metrics_)
print("Final estimator:", type(model.model_).__name__)

<class 'tuiml.workflow.Workflow'>
Metrics: {'accuracy_score': 0.9666666666666667, 'f1_score': 0.9665831244778613}
Final estimator: RandomForestClassifier


Algorithm hyperparameters go inside the `"params"` dict of the model spec. Loose keys are rejected; that one rule is what makes every spec safely serializable to JSON.

In [3]:
# Hyperparameters live under "params"
model = tuiml.train({
    "model": {"name": "RandomForestClassifier",
              "params": {"n_estimators": 100, "max_depth": 10}},
    "data": {"source": "iris", "target": "class"},
})

print("Accuracy:", model.metrics_.get("accuracy_score", "N/A"))

Accuracy: 0.9666666666666667


## 2. Cross-Validation

Evaluation options are grouped under the `"evaluation"` key. Set `{"cv": k}` to use k-fold cross-validation instead of a single train/test split; the reported metrics are averaged across folds (and the pipeline is still refitted on all the data afterwards).

In [4]:
# 5-fold cross-validation
model = tuiml.train({
    "model": {"name": "NaiveBayesClassifier"},
    "data": {"source": "iris", "target": "class"},
    "evaluation": {"cv": 5},
})

print("5-fold CV metrics:", model.metrics_)

5-fold CV metrics: {'cv_accuracy_score_mean': 0.9600000000000002, 'cv_accuracy_score_std': 0.024944382578492935, 'cv_f1_score_mean': 0.9591930613669744, 'cv_f1_score_std': 0.02521829936446811}


In [5]:
# 10-fold cross-validation (the standard in ML research)
model = tuiml.train({
    "model": {"name": "C45TreeClassifier"},
    "data": {"source": "iris", "target": "class"},
    "evaluation": {"cv": 10},
})

print("10-fold CV metrics:", model.metrics_)
print("Per-fold details:", model.cv_results_)

10-fold CV metrics: {'cv_accuracy_score_mean': 0.9533333333333334, 'cv_accuracy_score_std': 0.052068331172711015, 'cv_f1_score_mean': 0.952981092981093, 'cv_f1_score_std': 0.05561375901102343}
Per-fold details: {'scores': {'accuracy_score': [1.0, 1.0, 1.0, 1.0, 0.9333333333333333, 0.8666666666666667, 0.8666666666666667, 0.9333333333333333, 1.0, 0.9333333333333333], 'f1_score': [1.0, 1.0, 1.0, 1.0, 0.9259259259259259, 0.85, 0.8611111111111112, 0.9440559440559441, 1.0, 0.9487179487179486]}}


## 3. The Pipeline

The `"pipeline"` key is one ordered list of every step that runs *before* the model: imputation, scaling, encoding, feature generation, extraction, selection, resampling. Every step is a `{"name": ..., "params": {...}}` dict, the same shape as the model spec (`"params"` optional).

In [6]:
# Steps with default parameters need only a "name"
model = tuiml.train({
    "model": {"name": "SVC"},
    "data": {"source": "diabetes", "target": "class"},
    "pipeline": [{"name": "SimpleImputer"}, {"name": "StandardScaler"}],
    "evaluation": {"cv": 5},
})

print("With a pipeline:", model.metrics_)

With a pipeline: {'cv_accuracy_score_mean': 0.7643323996265172, 'cv_accuracy_score_std': 0.018869528208355697, 'cv_f1_score_mean': 0.6146467144814947, 'cv_f1_score_std': 0.03837104430446875}


In [7]:
# "params" for full control over each step
model = tuiml.train({
    "model": {"name": "KNearestNeighborsClassifier", "params": {"k": 5}},
    "data": {"source": "diabetes", "target": "class"},
    "pipeline": [
        {"name": "SimpleImputer", "params": {"strategy": "median"}},
        {"name": "MinMaxScaler"},
    ],
    "evaluation": {"cv": 5},
})

print("Custom pipeline:", model.metrics_)

Custom pipeline: {'cv_accuracy_score_mean': 0.710966810966811, 'cv_accuracy_score_std': 0.015141999148272198, 'cv_f1_score_mean': 0.547378542173882, 'cv_f1_score_std': 0.03767229011526063}


## 4. Presets

Presets are named pipelines. Pass the name straight under `"pipeline"` to apply a whole bundle in one key.

| Preset | Steps | Use case |
|--------|-------|----------|
| `"minimal"` | None | Clean data, no preprocessing needed |
| `"fast"` | Impute (most frequent) | Quick baseline with missing value handling |
| `"standard"` | Impute + MinMaxScale + OneHotEncode | General-purpose pipeline |
| `"full"` | Impute + StandardScale + OneHotEncode + SelectKBest | Full pipeline with feature selection |
| `"imbalanced"` | Impute + MinMaxScale + SMOTE | Class-imbalanced datasets |

In [8]:
# Standard preset: Impute + Scale + Encode
model = tuiml.train({
    "model": {"name": "LogisticRegression"},
    "data": {"source": "diabetes", "target": "class"},
    "pipeline": "standard",
    "evaluation": {"cv": 5},
})

print("Standard preset:", model.metrics_)

Standard preset: {'cv_accuracy_score_mean': 0.6510907393260335, 'cv_accuracy_score_std': 0.04435563443637152, 'cv_f1_score_mean': 0.44569805973159216, 'cv_f1_score_std': 0.04138886388124743}


In [9]:
# Full preset: includes feature selection
model = tuiml.train({
    "model": {"name": "RandomForestClassifier"},
    "data": {"source": "diabetes", "target": "class"},
    "pipeline": "full",
    "evaluation": {"cv": 5},
})

print("Full preset:", model.metrics_)

Full preset: {'cv_accuracy_score_mean': 0.6601561836855955, 'cv_accuracy_score_std': 0.027795085744255675, 'cv_f1_score_mean': 0.25000165295369803, 'cv_f1_score_std': 0.04242792532268909}


In [10]:
# Each preset is just a plain step list, so you can inspect it, or extend it
for name, steps in tuiml.PRESETS.items():
    labels = [s["name"] for s in steps]
    print(f"{name:12s} -> {labels or ['(no steps)']}")

# ...and compose: a preset plus your own extra step
custom = tuiml.PRESETS["standard"] + [{"name": "PCAExtractor", "params": {"n_components": 4}}]
print("\ncomposed  ->", [s["name"] for s in custom])

minimal      -> ['(no steps)']
fast         -> ['SimpleImputer']
standard     -> ['SimpleImputer', 'MinMaxScaler', 'OneHotEncoder']
full         -> ['SimpleImputer', 'StandardScaler', 'OneHotEncoder', 'SelectKBestSelector']
imbalanced   -> ['SimpleImputer', 'MinMaxScaler', 'SMOTESampler']

composed  -> ['SimpleImputer', 'MinMaxScaler', 'OneHotEncoder', 'PCAExtractor']


## 5. Feature Selection

Feature selection is not a separate key: a selector is just another step in the pipeline, and it goes last so it sees the cleaned, scaled features.

In [11]:
model = tuiml.train({
    "model": {"name": "NaiveBayesClassifier"},
    "data": {"source": "diabetes", "target": "class"},
    "pipeline": [
        {"name": "SimpleImputer"},
        {"name": "StandardScaler"},
        {"name": "SelectKBestSelector", "params": {"k": 4}},
    ],
    "evaluation": {"cv": 5},
})

print("With feature selection:", model.metrics_)

With feature selection: {'cv_accuracy_score_mean': 0.7526440879382056, 'cv_accuracy_score_std': 0.026382786223484726, 'cv_f1_score_mean': 0.619907009297235, 'cv_f1_score_std': 0.028698390758817808}


## 6. Everything Is a Spec (LLM-Friendly Format)

The whole run is one dict, and every component in it, the model and each pipeline step alike, is written the same way: `{"name": ..., "params": {...}}`. Bare name strings are rejected, and so are hyperparameters outside `"params"`. The payoff is that an entire training run is plain JSON: exactly what an LLM can emit without inventing keyword arguments.

In [12]:
# One shape for everything - ideal for LLM-generated configs
model = tuiml.train({
    "model": {"name": "RandomForestClassifier",
              "params": {"n_estimators": 50, "max_depth": 8}},
    "data": {"source": "iris", "target": "class"},
    "evaluation": {"cv": 5},
})

print("Dict-based model spec:", model.metrics_)

Dict-based model spec: {'cv_accuracy_score_mean': 0.9600000000000002, 'cv_accuracy_score_std': 0.024944382578492935, 'cv_f1_score_mean': 0.9588630532108793, 'cv_f1_score_std': 0.025783210306982992}


## 7. Config-Driven Workflows

Because the spec is one dict, it can live in a file: `train()` also accepts a path to a `.json` file holding it. The allowed keys are `model`, `data`, `target`, `features`, `pipeline`, `evaluation`, and `random_seed`; unknown keys raise `ValueError`. This is the preferred entry point for automated pipelines and LLM agents.

In [13]:
config = {
    "model": {"name": "XGBoostClassifier", "params": {"n_estimators": 50}},
    "data": {"source": "iris", "target": "class"},
    "pipeline": [{"name": "SimpleImputer"}, {"name": "MinMaxScaler"}],
    "evaluation": {"cv": 5},
}

model = tuiml.train(config)
print("Config-driven result:", model.metrics_)

Config-driven result: {'cv_accuracy_score_mean': 0.9466666666666667, 'cv_accuracy_score_std': 0.03399346342395189, 'cv_f1_score_mean': 0.9452627752937351, 'cv_f1_score_std': 0.03348547680913447}


In [14]:
# An LLM might generate this JSON config
llm_config = {
    "model": {"name": "SVC", "params": {"kernel": "rbf", "C": 1.0}},
    "data": {"source": "iris", "target": "class"},
    "pipeline": [
        {"name": "SimpleImputer", "params": {"strategy": "mean"}},
        {"name": "StandardScaler"},
    ],
    "evaluation": {"cv": 10, "metrics": ["accuracy_score", "f1_score"]},
}

model = tuiml.train(llm_config)
print("LLM config result:", model.metrics_)

# Round-trip: any fitted Workflow can export the spec that would reproduce it
print("\nRecovered spec:", model.to_config())

LLM config result: {'cv_accuracy_score_mean': 0.9600000000000002, 'cv_accuracy_score_std': 0.05333333333333332, 'cv_f1_score_mean': 0.9597150997150997, 'cv_f1_score_std': 0.056674315057102034}

Recovered spec: {'model': {'name': 'SVC'}, 'pipeline': [{'name': 'SimpleImputer'}, {'name': 'StandardScaler'}]}


## 8. `tuiml.Benchmark` - Compare Models

Compare multiple models across one or more datasets: configure the `Benchmark`, execute with `.run()`, and read the results off the instance, summary tables, rankings, and statistical tests. Model specs are strict `{"name", "params"}` dicts and dataset specs are always dicts like `{"source": ...}`. Shared preprocessing goes in `pipeline=`: the same step list a `train()` spec carries.

In [15]:
result = tuiml.Benchmark(
    models=[
        {"name": "RandomForestClassifier"},
        {"name": "NaiveBayesClassifier"},
        {"name": "C45TreeClassifier"},
        {"name": "SVC"},
        {"name": "KNearestNeighborsClassifier"},
    ],
    datasets=[{"source": "iris"}, {"source": "diabetes"}],
    pipeline=[{"name": "SimpleImputer"}, {"name": "StandardScaler"}],
    evaluation={"cv": 10},
    random_seed=42,
).run()

print(result.summary())

Benchmark: classification, 5 models x 2 datasets, seed 42

iris  (accuracy_score)
--------------------------------------------------
  RandomForestClassifier: 0.9467 ± 0.0526
  NaiveBayesClassifier: 0.9533 ± 0.0549
  C45TreeClassifier: 0.9533 ± 0.0450
  SVC: 0.9667 ± 0.0567  <- best
  KNearestNeighborsClassifier: 0.9467 ± 0.0526

diabetes  (accuracy_score)
--------------------------------------------------
  RandomForestClassifier: 0.7656 ± 0.0490  <- best
  NaiveBayesClassifier: 0.7525 ± 0.0534
  C45TreeClassifier: 0.7160 ± 0.0507
  SVC: 0.7618 ± 0.0284
  KNearestNeighborsClassifier: 0.7058 ± 0.0490


In [16]:
# Export as markdown table
print(result.to_markdown())

| Dataset | RandomForestClassifier | NaiveBayesClassifier | C45TreeClassifier | SVC | KNearestNeighborsClassifier |
|---|---|---|---|---|---|
| iris | 0.9467 ± 0.0526 | 0.9533 ± 0.0549 | 0.9533 ± 0.0450 | **0.9667 ± 0.0567** | 0.9467 ± 0.0526 |
| diabetes | **0.7656 ± 0.0490** | 0.7525 ± 0.0534 | 0.7160 ± 0.0507 | 0.7618 ± 0.0284 | 0.7058 ± 0.0490 |


## 9. Predict & Evaluate

`train()` hands back a fitted pipeline, so prediction and evaluation are methods on it. New rows pass through every fitted transformation step before reaching the model, so there is no chance of forgetting to scale your test data.

In [17]:
# Train a model
model = tuiml.train({
    "model": {"name": "RandomForestClassifier"},
    "data": {"source": "iris", "target": "class"},
})

# Predict on new samples - the fitted pipeline travels with the model
new_data = np.array([
    [5.1, 3.5, 1.4, 0.2],
    [6.7, 3.0, 5.2, 2.3],
])
print("Predictions:", model.predict(new_data))

Predictions: [0 2]


In [18]:
# Evaluate against known labels
from tuiml.datasets import load_iris
from tuiml.evaluation import train_test_split
from tuiml.preprocessing import StandardScaler
from tuiml.algorithms.trees import RandomForestClassifier

X, y = load_iris()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# A Workflow fits like any estimator when you hand it arrays.
# Workflow is the object door: its steps are configured instances.
pipe = tuiml.Workflow([
    StandardScaler(),
    RandomForestClassifier(n_estimators=100),
])
pipe.fit(X_train, y_train)

# Full metric dict, or a single number with .score()
print("Evaluation:", pipe.evaluate(X_test, y_test))
print("Score:     ", pipe.score(X_test, y_test))

Evaluation: {'accuracy_score': 1.0, 'f1_score': 1.0}
Score:      1.0


## 10. Save & Load

`model.save(path)` writes the **whole pipeline**, every fitted transformation plus the final estimator, to one file. `Workflow.load(path)` brings it back ready to predict.

In [19]:
import os
from tuiml import Workflow

# Train
model = tuiml.train({
    "model": {"name": "NaiveBayesClassifier"},
    "data": {"source": "iris", "target": "class"},
    "pipeline": [{"name": "StandardScaler"}],
    "evaluation": {"cv": 5},
})
print("Original metrics:", model.metrics_)

# Save the entire pipeline as one file
model.save("iris_nb_model.pkl")

# Load it back
loaded = Workflow.load("iris_nb_model.pkl")

# Predict with the loaded pipeline
sample = np.array([[5.1, 3.5, 1.4, 0.2]])
print("Loaded pipeline steps:", list(loaded.named_steps))
print("Loaded model prediction:", loaded.predict(sample))

# Clean up
os.remove("iris_nb_model.pkl")

Original metrics: {'cv_accuracy_score_mean': 0.9600000000000002, 'cv_accuracy_score_std': 0.024944382578492935, 'cv_f1_score_mean': 0.9591930613669744, 'cv_f1_score_std': 0.02521829936446811}
Loaded pipeline steps: ['standardscaler', 'naivebayesclassifier']
Loaded model prediction: [0]


## 11. Algorithm Discovery

Three functions help you explore what TuiML offers.

In [20]:
# List all classifiers
classifiers = tuiml.list_algorithms(type="classifier")
print(f"{len(classifiers)} classifiers available:")
for algo in classifiers[:8]:  # Show first 8
    print(f"  {algo['name']}")

89 classifiers available:
  MyVariant
  MyVariant_v1_0_1
  NaiveBayesClassifier
  NaiveBayesMultinomialClassifier
  CategoricalNBClassifier
  BayesianNetworkClassifier
  DecisionStumpClassifier
  C45TreeClassifier


In [21]:
# Describe a specific algorithm and its parameters
info = tuiml.describe_algorithm("RandomForestClassifier")
print(f"Name: {info['name']}")
print(f"Type: {info['type']}")
print("Parameters:")
for param, schema in info["parameters"].items():
    print(f"  {param}: {schema.get('type', '?')} = {schema.get('default', '?')}")

Name: RandomForestClassifier
Type: ComponentType.CLASSIFIER
Parameters:
  n_estimators: integer = 100
  max_features: ['string', 'integer', 'number'] = sqrt
  max_depth: integer = None
  min_samples_split: integer = 2
  min_samples_leaf: integer = 1
  bootstrap: boolean = True
  oob_score: boolean = False
  random_state: integer = None
  n_jobs: integer = 1
  criterion: string = gini


In [22]:
# Search by keyword
results = tuiml.search_algorithms("boost")
print("Algorithms matching 'boost':")
for algo in results:
    print(f"  {algo['name']}")

Algorithms matching 'boost':
  AdaBoostClassifier
  AdaBoostRegressor
  CatBoostClassifier
  CatBoostRegressor
  LogitBoostClassifier
  XGBoostClassifier
  XGBoostRegressor
  sklearn.AdaBoostClassifier
  sklearn.AdaBoostRegressor
  sklearn.GradientBoostingClassifier
  sklearn.GradientBoostingRegressor
  sklearn.HistGradientBoostingClassifier
  sklearn.HistGradientBoostingRegressor
  DecisionStumpClassifier
  LogisticModelTreeClassifier
  SimpleLogisticRegression


## 12. API Discovery for LLMs

`tuiml.agent.get_tools_for_llm()` returns machine-readable tool schemas for every TuiML capability. An LLM agent calls this to discover what TuiML can do and what arguments each tool takes: the same 30 tools the MCP server exposes.

In [23]:
from tuiml.agent import get_tools_for_llm

tools = get_tools_for_llm()
print(f"{len(tools)} tools available to an agent")

# Schema for a single tool
train_tool = next(t for t in tools if t["name"] == "tuiml_train")
print("\ntuiml_train schema:")
print(f"  Description: {train_tool['description'][:80]}...")
print(f"  Parameters:  {list(train_tool['inputSchema']['properties'])}")

30 tools available to an agent

tuiml_train schema:
  Description: Train a machine learning model with evaluation. Two evaluation modes:
1. Holdout...
  Parameters:  ['algorithm', 'data', 'target', 'features', 'preprocessing', 'feature_selection', 'cv', 'test_size', 'metrics', 'preset', 'algorithm_params', 'save_path', 'random_seed', 'stage', 'stage_kwargs', 'model_id', 'model_path']


In [24]:
# The full catalogue an agent sees at connection time
for tool in tools:
    print(f"  {tool['name']:26s} - {tool['description'].splitlines()[0][:60]}")

  tuiml_train                - Train a machine learning model with evaluation. Two evaluati
  tuiml_predict              - Make predictions using a trained model on new data. Supports
  tuiml_evaluate             - Evaluate a trained model on test data and compute metrics.
  tuiml_benchmark            - Compare multiple algorithms on one or more datasets with cro
  tuiml_upload_data          - Register a dataset for use with other TuiML tools. Provide e
  tuiml_save_model           - Copy a trained model to a custom path. Use this when the use
  tuiml_serve_model          - Start a REST API server to serve a trained model for predict
  tuiml_stop_server          - Stop a running model serving API server.
  tuiml_server_status        - Get status of running model serving API servers.
  tuiml_plot                 - Generate a visualization/plot for model analysis. Returns th
  tuiml_profile_data         - Inspect a dataset before training, shape, dtypes, missing va
  tuiml_generate_data 